# Attention as a write into the residual

**Previous:** [03_residual_stream](03_residual_stream.ipynb)  
**Home:** [../00_START_HERE.ipynb](../00_START_HERE.ipynb)  
**Next:** [05_mlp](05_mlp.ipynb) · depth: [09_evolution_through_transformer](09_evolution_through_transformer.ipynb)

**Kernel:** `CXR local Qwen (faiss_gpu1)`  
**Status:** executable (calls `scripts/` — same as CLI)

---

## 1. Why am I learning this?

Attention produces a same-width vector that is **added** to the residual.

## 2. Mental model

Site `attn` vs `block` — two cameras on related tensors.

## 3. Clinical question

How does attn L2 compare to block L2 at **L0**? (Full L0–L27 depth → [09_evolution](09_evolution_through_transformer.ipynb); do not jump straight to L20.)

## 4. Prediction

Both non-zero at L0. Depth differences come later — evolution lesson, not this one.


## 5. Minimal Python — setup + load (run once)


In [30]:
# Shared bootstrap — run this first in every executable notebook
import sys
from pathlib import Path

# notebooks/_lib regardless of how deep this notebook sits
_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "_lib"))
        break
    if (_p / "notebooks" / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "notebooks" / "_lib"))
        break
else:
    raise RuntimeError("Cannot find notebooks/_lib/cxr_boot.py — open Jupyter with notebooks/ as root")

import cxr_boot
ctx = cxr_boot.setup(layer=20, max_new=24, load_model=True)
model, tok = ctx["model"], ctx["tok"]
NOTE, LAYER, MAX_NEW = ctx["NOTE"], ctx["LAYER"], ctx["MAX_NEW"]
PROMPT_A, PROMPT_B, PROMPT_TEST = ctx["PROMPT_A"], ctx["PROMPT_B"], ctx["PROMPT_TEST"]
look, intervene, process = ctx["look"], ctx["intervene"], ctx["process"]
print("NOTE:", NOTE)
print("LAYER:", LAYER, "ready")


BACKEND  REUSE — not calling from_pretrained; inspecting live weights
  pid=4097531  class=Qwen2ForCausalLM
  name=Qwen/Qwen2.5-7B-Instruct
  torch_dtype=torch.float16  training=False
  blocks=28  hidden=3584  using layer 20
  tokenizer=Qwen2TokenizerFast  vocab=151643  pad=151643  eos=151645
  hf_device_map (accelerate placement):
    model.embed_tokens: 0
    model.layers.0: 0
    model.layers.1: 0
    model.layers.2: 0
    model.layers.3: 0
    model.layers.4: 0
    model.layers.5: 0
    model.layers.6: 0
    model.layers.7: 0
    model.layers.8: 0
    model.layers.9: 0
    model.layers.10: 0
    model.layers.11: 0
    model.layers.12: 0
    model.layers.13: 0
    model.layers.14: 0
    model.layers.15: 0
    model.layers.16: 0
    model.layers.17: 0
    model.layers.18: 0
    model.layers.19: 0
    model.layers.20: 0
    model.layers.21: 0
    model.layers.22: 0
    model.layers.23: 0
    model.layers.24: 0
    model.layers.25: 0
    model.layers.26: 0
    model.layers.27: cpu
    

Yes. Let's treat this as **Lesson 0: starting the laboratory** and separate **what each piece of code does** from **what every output field means**.

The single most important point is: **this cell does not analyze the note yet.** It prepares the model, tokenizer, note, and MI tools so later cells can run experiments.

## 1. Finding your shared notebook library

```python
import sys
from pathlib import Path
```

`Path` lets Python work with folders/files cleanly. `sys` gives access to Python's module search path.

Then:

```python
_here = Path.cwd().resolve()
```

`Path.cwd()` asks:

> What directory is this notebook currently running from?

Suppose that's:

```text
/project/notebooks/01_foundations/
```

`.resolve()` converts it to a complete absolute path.

Then:

```python
for _p in [_here, *_here.parents]:
```

means:

> Start in the current directory, then walk upward through every parent directory.

For example:

```text
/project/notebooks/01_foundations/
            ↓
/project/notebooks/
            ↓
/project/
            ↓
/
```

At each location, you're looking for your shared library.

---

## 2. Finding `cxr_boot.py`

First:

```python
if (_p / "_lib" / "cxr_boot.py").is_file():
```

asks:

> Is there an `_lib/cxr_boot.py` here?

If yes:

```python
sys.path.insert(0, str(_p / "_lib"))
break
```

adds that `_lib` directory to Python's import search path.

So Python can subsequently do:

```python
import cxr_boot
```

The second condition:

```python
if (_p / "notebooks" / "_lib" / "cxr_boot.py").is_file():
```

handles another possible directory structure:

```text
project/
├── ...
└── notebooks/
    └── _lib/
        └── cxr_boot.py
```

This is what makes your bootstrap reusable even when notebooks are nested several folders deep.

---

## 3. The error condition

```python
else:
    raise RuntimeError(...)
```

The `else` belongs to the `for` loop.

It means:

> I searched the current directory and all its parents and never found `cxr_boot.py`.

Rather than letting later code fail mysteriously, it stops immediately and tells you what's wrong.

So this entire first section is basically:

```text
Where am I?
    ↓
Find _lib/cxr_boot.py
    ↓
Tell Python where it is
    ↓
Now I can import it
```

It has nothing specifically to do with transformers yet.

---

# 4. Importing your CXR bootstrap

Now:

```python
import cxr_boot
```

loads your shared bootstrap module.

Think of `cxr_boot.py` as the **laboratory setup assistant**.

Instead of every notebook containing hundreds of lines for:

```text
find model
load/reuse model
find tokenizer
configure device
load prompts
import LOOK
import INTERVENE
import PROCESS
set note
set layer
...
```

you've centralized that work in `cxr_boot`.

---

# 5. This is the important line

```python
ctx = cxr_boot.setup(
    layer=20,
    max_new=24,
    load_model=True
)
```

You're telling the bootstrap three things.

### `layer=20`

Your default experimental layer is:

```text
L20
```

It doesn't mean anything has happened to L20 yet.

You're simply saying:

> When an experiment needs a default layer, use transformer block 20.

### `max_new=24`

When you generate text, allow up to:

```text
24 new tokens
```

Again, nothing is being generated yet.

### `load_model=True`

This says:

> Make the model available to this notebook.

But, importantly, your backend realizes that Qwen is **already loaded**, so it doesn't create another copy.

We'll see that in the output.

---

# 6. What is `ctx`?

`ctx` is essentially a Python dictionary containing your laboratory equipment.

You unpack some of it here:

```python
model, tok = ctx["model"], ctx["tok"]
```

So:

```text
model
```

is the actual neural network — Qwen2.5-7B-Instruct.

And:

```text
tok
```

is its tokenizer.

Then:

```python
NOTE, LAYER, MAX_NEW = (
    ctx["NOTE"],
    ctx["LAYER"],
    ctx["MAX_NEW"]
)
```

retrieves your default:

```text
NOTE
LAYER
MAX_NEW
```

Then:

```python
PROMPT_A, PROMPT_B, PROMPT_TEST = ...
```

retrieves the prompts your experiments can use.

And:

```python
look, intervene, process = ...
```

is particularly important for your MI curriculum.

Conceptually:

```text
LOOK
  │
  └── Observe what's happening inside the model

INTERVENE
  │
  └── Change something inside the model

PROCESS
  │
  └── Supporting/processing operations
```

That separation is excellent for learning MI because **observation and intervention answer different questions**.

---

# 7. Finally

```python
print("NOTE:", NOTE)
print("LAYER:", LAYER, "ready")
```

just displays your current note and selected layer.

Now we can decode the output.

---

# 8. `BACKEND REUSE`

Your output starts:

```text
BACKEND REUSE — not calling from_pretrained;
inspecting live weights
```

This is important.

Normally, Hugging Face might load Qwen using something like:

```python
from_pretrained(...)
```

Your system discovered:

> Qwen is already sitting in memory.

Therefore it is **reusing the existing model**.

That's exactly what you want for notebooks.

Otherwise Notebook 1 could load ~14 GiB, Notebook 2 could try loading another ~14 GiB, etc.

Instead:

```text
                   MODEL BACKEND
              Qwen2.5-7B-Instruct
                       │
              already in memory
                       │
          ┌────────────┼────────────┐
          ▼            ▼            ▼
      Notebook 1   Notebook 2   Notebook 3
```

---

# 9. `pid=4097531`

```text
pid=4097531
```

PID means **Process ID**.

Linux assigned the running backend process the identifier:

```text
4097531
```

This isn't an MI concept. It's mostly useful for debugging and confirming that notebooks are talking to the same running process.

---

# 10. `class=Qwen2ForCausalLM`

```text
class=Qwen2ForCausalLM
```

This is the Python/Hugging Face model class.

`CausalLM` means **causal language model**.

In simplified terms:

```text
Previous tokens
      ↓
Transformer
      ↓
Predict next token
```

---

# 11. Model name

```text
name=Qwen/Qwen2.5-7B-Instruct
```

This tells you exactly which model you're studying:

**Qwen2.5-7B-Instruct.**

The "Instruct" version has been adapted to follow instructions/conversations rather than being only a base next-token model.

---

# 12. `torch_dtype=torch.float16`

```text
torch_dtype=torch.float16
```

Your weights use 16-bit floating-point representation.

Very roughly:

```text
one parameter ≈ 2 bytes
```

That's going to explain the model's memory usage later.

---

# 13. `training=False`

```text
training=False
```

The model is not currently in training mode.

You're using it for **inference/analysis**.

That's what you generally want for your MI experiments.

---

# 14. `blocks=28`

Now we're getting into actual transformer architecture:

```text
blocks=28
```

Qwen has **28 transformer blocks** in this model.

Python numbers them starting from zero:

```text
L0
L1
L2
L3
...
L20
...
L26
L27
```

So there are 28 total:

$$
0 \rightarrow 27
$$

This distinction matters enormously because, as we discussed previously:

```text
L20
```

means **Layer 20**,

whereas:

```text
L2=10.43
```

in your other experiment meant **L2 vector norm**.

They are completely different uses of `L`.

---

# 15. `hidden=3584`

This is one of the most important numbers in the entire output:

```text
hidden=3584
```

Your residual stream width is **3,584 dimensions**.

At each token position, the model carries something like:

$$
r \in \mathbb{R}^{3584}
$$

Meaning a residual vector contains:

```text
[x₁, x₂, x₃, ............, x₃₅₈₄]
```

So when you previously captured:

```python
bucket["rin"][0, -1, :]
```

you were extracting one of these **3,584-dimensional vectors**.

This is central to both MI and RepEng.

---

# 16. `using layer 20`

```text
using layer 20
```

That's simply reflecting:

```python
layer=20
```

from your bootstrap.

You selected:

```text
28 transformer blocks

L0
 ↓
L1
 ↓
...
 ↓
L19
 ↓
╔═══════╗
║  L20  ║ ← selected
╚═══════╝
 ↓
L21
 ↓
...
 ↓
L27
```

Again, it doesn't mean you've analyzed L20 yet.

---

# 17. Tokenizer

```text
tokenizer=Qwen2TokenizerFast
```

The tokenizer converts your text into token IDs.

Your sentence:

```text
Patient received FOLFOX.
Disease progressed.
FOLFOX was discontinued.
```

does **not** enter the transformer directly as English words.

It goes:

```text
TEXT
 ↓
TOKENIZER
 ↓
token IDs
 ↓
EMBEDDINGS
 ↓
3584-dimensional representations
 ↓
TRANSFORMER
```

Later, I strongly recommend making the actual tokens visible in your notebook.

---

# 18. `vocab=151643`

```text
vocab=151643
```

The model's configured tokenizer vocabulary contains about **151,643 token entries**.

Ultimately the language model assigns scores to possible next tokens from its output vocabulary.

---

# 19. `pad` and `eos`

```text
pad=151643
eos=151645
```

These are special token IDs.

`pad` is the padding token.

`eos` means:

**End Of Sequence.**

The numerical IDs themselves aren't particularly important for learning MI right now.

---

# 20. The huge `hf_device_map`

Now you see:

```text
hf_device_map (accelerate placement):
```

This tells you **where each piece of the model physically lives**.

For example:

```text
model.layers.0: 0
```

means:

> Transformer Layer 0 is on CUDA GPU 0.

Likewise:

```text
model.layers.20: 0
```

means:

> Layer 20 is on GPU 0.

Almost all your transformer blocks are there:

```text
GPU 0
──────────────────
embedding
L0
L1
L2
...
L20 ← your target
...
L26
```

But then:

```text
model.layers.27: cpu
model.norm: cpu
model.rotary_emb: cpu
lm_head: cpu
```

means those components have been assigned/offloaded to the CPU.

So approximately:

```text
RTX 3090                       CPU/RAM

Embedding                      L27
L0                             Final norm
L1                             Rotary component
...                            LM head
L20
...
L26
```

This is mainly **engineering information**, not MI theory.

But it's useful because you have a 24-GB GPU and you're fitting a large model into available memory.

---

# 21. `device_map counts`

```text
device_map counts:
{'0': 28, 'cpu': 4}
```

This summarizes the preceding list.

Most mapped components are assigned to:

```text
GPU 0
```

and four mapped entries are assigned to:

```text
CPU
```

Don't confuse this count with "28 layers + 4 layers." These are **device-map entries/components**, not all necessarily transformer blocks.

---

# 22. Parameter tensors

```text
param tensors by device:
{'cuda:0': 325, 'meta': 14}
```

The model's parameters aren't one giant tensor.

They're stored as many tensors:

```text
attention weights
MLP weights
normalization weights
embedding weights
...
```

325 parameter tensors are currently represented on CUDA 0 in this inspection.

The `meta` entries are related to Hugging Face/Accelerate's dispatch/offloading machinery; a meta tensor is a placeholder representation rather than ordinary allocated model data.

For your MI curriculum, I'd mark this:

> **Backend detail — understand later.**

You don't need this to understand residual streams, attention, MLPs, steering, SAEs, patching, etc.

---

# 23. Parameter data types

```text
param dtypes:
{'torch.float16': 339}
```

339 parameter tensors inspected are FP16.

Again:

```text
FP16
=
16 bits
=
2 bytes per value
```

---

# 24. Model parameter count

```text
params=7.616B
```

Your model contains approximately:

$$
7.616 \text{ billion parameters}
$$

These are the learned weights.

Important distinction:

```text
PARAMETERS
≈ 7.616 billion
Learned model weights
Mostly fixed during your MI experiments

vs.

ACTIVATIONS
Generated when your particular NOTE passes through model
Change from input to input
```

Your mechanistic work is heavily concerned with the second category.

---

# 25. `weight_GiB=14.19`

```text
weight_GiB=14.19
```

Your model weights occupy roughly **14.19 GiB**.

That makes sense because:

$$
7.616B \times 2\text{ bytes}
$$

is approximately 15.2 billion bytes, or about 14.2 GiB.

So those numbers cross-check nicely.

---

# 26. CUDA allocated memory

```text
cuda_alloc_GiB=12.74
```

PyTorch currently has approximately:

**12.74 GiB of GPU memory actively allocated.**

This includes model tensors and other CUDA allocations.

---

# 27. CUDA reserved

```text
cuda_reserved_GiB=13.79
```

PyTorch keeps some GPU memory reserved so it can reuse it efficiently rather than constantly asking CUDA for memory.

Therefore:

```text
allocated ≠ reserved
```

That's normal.

---

# 28. GPU free

```text
cuda0 free_GiB=8.62
```

Approximately **8.62 GiB remains free** on GPU 0 at that moment.

That's useful for your MI experiments because hooks, saved activations, SAE calculations, patching, etc. can consume additional memory.

---

# 29. Total GPU memory

```text
total_GiB=23.56
```

That's your approximately 24-GB GPU expressed in GiB.

So roughly:

```text
GPU MEMORY
─────────────────────────────

Total              23.56 GiB
Free                8.62 GiB
PyTorch allocated  12.74 GiB
PyTorch reserved   13.79 GiB
```

Don't try to make those categories add up directly; "reserved" includes memory PyTorch manages and overlaps conceptually with allocated memory.

---

# 30. Your NOTE

Finally:

```text
NOTE:
Patient received FOLFOX.
Disease progressed.
FOLFOX was discontinued.
```

This comes from:

```python
NOTE = ctx["NOTE"]
```

This is your experimental input.

Nothing in this bootstrap output tells us yet **how Qwen represented this sentence**.

That's an important distinction.

---

# 31. `LAYER: 20 ready`

And:

```text
LAYER: 20 ready
```

comes directly from:

```python
print("LAYER:", LAYER, "ready")
```

It means:

> Your default experimental layer has been set to L20 and the environment is ready.

It does **not** mean:

> L20 detected FOLFOX failure.

It does **not** mean:

> L20 contains the temporal representation.

It does **not** mean:

> L20 caused the answer.

Those require experiments.

---

## So reduce the entire output to four things

Most of that giant output is engineering diagnostics. For understanding MI, I'd mentally compress it to:

```text
MY LAB
══════════════════════════════════

MODEL
Qwen2.5-7B-Instruct
7.616B parameters
FP16

ARCHITECTURE
28 transformer blocks: L0–L27
Residual width: 3584

EXPERIMENT
Target layer: L20
Input:
"Patient received FOLFOX.
 Disease progressed.
 FOLFOX was discontinued."

COMPUTE
Most model layers: RTX 3090
Some final components: CPU
~8.6 GiB GPU free

STATUS
Model reused ✓
Tokenizer ready ✓
LOOK ready ✓
INTERVENE ready ✓
Experiment not run yet
```

And there's a useful hierarchy for your notebook learning:

```text
BOOTSTRAP                 ← you are here
   │
   ▼
TOKENIZATION
   │
   ▼
EMBEDDING
   │
   ▼
RESIDUAL STREAM
   │
   ├── Attention
   │
   └── MLP
   │
   ▼
ACTIVATIONS
   │
   ▼
REPRESENTATIONS
   │
   ├── Difference vectors
   ├── Probes
   └── SAE features
   │
   ▼
CAUSAL MI
   │
   ├── Ablation
   ├── Patching
   └── Circuits
   │
   ▼
REPRESENTATION ENGINEERING
   │
   ├── Directions
   └── Steering
   │
   ▼
BEHAVIOR
```

Your previous Layer-0 experiment was already the next step down this ladder: it showed you the **residual stream entering the block, attention writing to it, the MLP writing to it, and the resulting residual leaving the block**. This bootstrap is simply what makes all those subsequent experiments possible.


## 6. Run


In [ ]:
# Attention write at L0 only — do not jump L0 → L20 here.
# Full depth (all 28 layers + curves) is 09_evolution_through_transformer.ipynb
look.cmd_sites(model, tok, layer=0, text=NOTE)

# Optional: peek CXR-prior zoom once you already understand L0
print(f"\n=== layer {LAYER} (CXR prior zoom only — not justified yet) ===")
look.cmd_sites(model, tok, layer=LAYER, text=NOTE)
print("\n→ For L0…L27 evolution + magnitude curves, open 09_evolution_through_transformer.ipynb")


> **Pedagogy update (2026-09-18):** Prefer **[09_evolution_through_transformer](09_evolution_through_transformer.ipynb)** for L0–L27 magnitude/representation sweeps. This notebook’s L0↔L20 peek is a **convenience contrast**, not evidence that L20 is special.

Good. This experiment is doing something different from the detailed Layer-0 hook experiment: it gives you a **quick snapshot of the sizes of three activation vectors at two different depths in the transformer**.

Your code:

```python
for layer in (0, LAYER):
    print(f"\n=== layer {layer} ===")
    look.cmd_sites(model, tok, layer=layer, text=NOTE)
```

Since `LAYER = 20`, it runs the same measurement twice:

```text
NOTE
 │
 ▼
L0   ← measure here
 │
 ▼
...
 │
 ▼
L20  ← measure here
 │
 ▼
...
L27
```

And in both cases it is examining the **last-token position**.

### What the three measurements mean

At Layer 0 you got:

```text
L0 block   dim=3584   L2=10.4349
L0 attn    dim=3584   L2=7.3859
L0 mlp     dim=3584   L2=4.5926
```

`dim=3584` means each captured object is a vector with 3,584 numbers.

`L2` means the **length/magnitude of that vector**, not "Layer 2."

So Layer 0 says:

```text
                       magnitude

Attention write  ─────── 7.39
MLP write        ────     4.59
Block output     ────────── 10.43
```

This agrees almost exactly with your previous detailed experiment:

```text
Previous                   This run

attention  7.3870          7.3859
MLP        4.5942          4.5926
block     10.4372         10.4349
```

The tiny differences are negligible numerical/run variation.

---

## Now Layer 20

This is where it becomes interesting:

```text
L20 block   dim=3584   L2=114.6233
L20 attn    dim=3584   L2=21.6076
L20 mlp     dim=3584   L2=36.9287
```

The dimensions have **not changed**:

```text
L0      3584 dimensions
         ↓
...
         ↓
L20     3584 dimensions
```

That's important. The residual stream remains 3,584-dimensional throughout these transformer blocks.

What's changing is the **values inside that vector** and therefore its magnitude.

Compare:

| Site            | Layer 0 |   Layer 20 |
| --------------- | ------: | ---------: |
| Block output    |   10.43 | **114.62** |
| Attention write |    7.39 |  **21.61** |
| MLP write       |    4.59 |  **36.93** |

So by Layer 20, the residual stream has accumulated much larger activations.

Conceptually, information has been repeatedly written into the residual stream:

```text
Initial representation
        │
        ▼
       L0
 Attention + MLP
        │
        ▼
       L1
 Attention + MLP
        │
        ▼
       L2
 Attention + MLP
        │
       ...
        │
        ▼
       L20
```

Each layer reads the current residual stream, performs computations, and writes additional information back into it.

So it is not surprising that the magnitude at L20 can be much larger than at L0.

### But there's a very important thing you cannot conclude

It is tempting to look at:

```text
L0 block  = 10.4
L20 block = 114.6
```

and say:

> "The model understands the note about 11× better at L20."

**No.**

Or:

> "The FOLFOX-failure representation has become stronger."

Also **not established**.

All you've measured is **vector magnitude**.

Imagine two arrows:

```text
                    ↑
                    │
                    │ 114
                    │
                    │
                    │
                    ●

       ───────►
          10
```

One arrow is longer, but they point in completely different directions.

An L2 norm tells you:

> **How big is this activation?**

It doesn't tell you:

> **What does this activation represent?**

That distinction is central to what you're learning.

---

## There's another subtlety here

Don't interpret:

```text
attention = 21.61
MLP       = 36.93
block     = 114.62
```

as:

$$
21.61 + 36.93 = 114.62
$$

because these are **norms of vectors**, not scalar contributions.

Also, unlike your previous detailed Layer-0 experiment, this quick `cmd_sites` output does **not show `resid_in`**.

At L20, a large residual is already arriving from L19.

Conceptually:

```text
               L20

          resid_in
      already quite large
               │
               │
       ┌───────┴───────┐
       ▼
   Attention
 write ||a||=21.61
       │
       ▼
residual + attention
       │
       ▼
      MLP
 write ||m||=36.93
       │
       ▼
   resid_out
 ||r|| = 114.62
```

So the 114.62 isn't being created from nothing by L20's attention and MLP. L20 inherits the accumulated residual from the preceding layers.

---

## And that's why these two experiments complement each other

Your first detailed code taught you **how one transformer block works**:

$$
r_{out} \approx r_{in}+a+m
$$

Your new code teaches you:

> **The same basic structure exists at different depths, but the activation magnitudes can be dramatically different.**

So in notebook-learning terms, you've now established two things:

**Experiment 1 — Anatomy**

> What happens inside one transformer block?

You demonstrated residual → attention write → add → MLP write → add.

**Experiment 2 — Depth**

> Does the internal state look numerically the same at the beginning and much later in the transformer?

No. Same dimensionality (**3584**), very different vector magnitudes.

The natural next experiment is therefore not another norm. It's to ask:

> **How does the *direction* of the residual representation change from L0 to L20?**

That's where cosine similarity becomes useful. Norm tells you **how big the arrow is**; cosine similarity begins telling you **whether two arrows point in similar directions**.


## 7–8. What happened / What did I learn?

_(fill after you run)_

## 9. Claim boundary

✓ We measured attn-site activations at two layers.

✗ Not a circuit; not head-level; not causal.

## 10. CXR connection

`look.py --mode sites`

## 11. Questions

## 12. Revision notes

| Date | Change |
|------|--------|
| | |
